# `05_non_bicycle_route_ways` per Spatial Unit

This notebook derives three tables: `non_bicycle_route_ways_per_municipality`, `non_bicycle_route_ways_per_province`, and `non_bicycle_route_ways_per_h3_cell`. Each table links the ways making up the cyclable network outside designated bicycle route relations (`non_bicycle_route_ways`) to a spatial unit, with one row per way–unit pair and the length of the way portion falling within that unit.

The notebook proceeds in three steps, one per spatial unit. In each step, `non_bicycle_route_ways` is spatially joined to the target unit boundaries, municipalities, provinces, and H3 grid cells respectively. For every way, the join identifies all spatial units the way intersects and computes the length of the portion falling within each unit (`clipped_length_meters`), using geometric intersection in the projected coordinate system EPSG:28992 which preserves distances in meters. The clipped geometry itself is not retained; only the length derived from it is stored, as the original way geometry is already available in `non_bicycle_route_ways` and the clipped length is the quantity needed for metric computation in `06_non_bicycle_route_metrics`.

A way lying entirely within a single spatial unit produces one row, with `clipped_length_meters` equal to its full length. A way crossing multiple spatial units produces one row per intersected unit, with `clipped_length_meters` reflecting the length of each portion. In both cases the sum of `clipped_length_meters` across all rows for a given way equals its original total length.

---
 
## Table of Contents

1. [Setup](#1-setup)
2. [Core clipping function](#2-core-clipping-function)
3. [Municipalities](#3-municipalities)
4. [Provinces](#4-provinces)
5. [H3 grid cells](#5-h3-grid-cells)
6. [Validation](#6-validation)

---
 
## 1. Setup

- ### Import relevant libraries

In [1]:
from IPython.utils import io

- ### Execute relevant notebooks and load variables into current session

In [2]:
# Access variables: 
with io.capture_output() as captured:
    %run /home/vbo226/00_non_bicycle_route_ways.ipynb # non_bicycle_route_ways
    %run /home/vbo226/03_boundaries_population.ipynb # municipalities, provinces, h3_cells

In [3]:
# For each spatial unit (municipality, province and h3 cell), convert GeoDataFrame into Arrow for easy ingestion into DuckDB 
municipality_arrow = municipalities.to_arrow()
provinces_arrow = provinces.to_arrow()
h3_cells_arrow = h3_cells.to_arrow()

In [4]:
pd.reset_option('^display')

---

## 2. Core clipping function
 
The function below clips ways from any source table to any spatial unit table, computing the intersection length in EPSG:28992. The ways table name and geometry column are passed as parameters, making the function reusable across both this notebook and `05_bicycle_route_m_ways_distinct_per_spatial_unit`.

In [5]:
def clip_ways_to_spatial_unit(spatial_unit_table, ways_table, geom_col):
    return duckdb.sql(f"""
    SELECT 
        s.*, 
        w.*, 
        ROUND(ST_Length(ST_Transform(ST_Intersection(w.{geom_col}, s.geometry),'EPSG:4326', 'EPSG:28992', always_xy := true)), 2) AS clipped_length_meters
    FROM {ways_table} w 
    JOIN {spatial_unit_table} s
      ON ST_Intersects(w.{geom_col}, s.geometry)
    """)

---

## 3. Municipalities

In [6]:
# Clip non-route ways to municipal boundaries
non_bicycle_route_ways_per_municipality = clip_ways_to_spatial_unit(
    spatial_unit_table='municipality_arrow',
    ways_table='non_bicycle_route_ways',
    geom_col='geometry'
)
 
# Row count after clipping
non_bicycle_route_ways_per_municipality.count('*')

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       252296 │
└──────────────┘

---

## 4. Provinces

In [7]:
# Clip non-route ways to provincial boundaries
non_bicycle_route_ways_per_province = clip_ways_to_spatial_unit(
    spatial_unit_table='provinces_arrow',
    ways_table='non_bicycle_route_ways',
    geom_col='geometry'
)
 
# Row count after clipping
non_bicycle_route_ways_per_province.count('*')

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       250093 │
└──────────────┘

---

In [8]:
# Clip non-route ways to H3 grid cells
non_bicycle_route_ways_per_h3_cell = clip_ways_to_spatial_unit(
    spatial_unit_table='h3_cells_arrow',
    ways_table='non_bicycle_route_ways',
    geom_col='geometry'
)
 
# Row count after clipping
non_bicycle_route_ways_per_h3_cell.count('*')

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       297730 │
└──────────────┘

---

## 6. Validation
 
To validate the spatial intersection process, the number of resulting segments is compared to the original number of distinct non-bicycle-route ways.
 
After intersection, the number of records varies by spatial partitioning level: `252 296` for municipalities, `250 093` for provinces, and `297 730` for H3 grid cells.

In [9]:
original_count = duckdb.sql("SELECT COUNT(*) FROM non_bicycle_route_ways").fetchone()[0]

original_length_km = duckdb.sql("""
    SELECT ROUND(SUM(length_nl_meters) / 1000, 3) FROM non_bicycle_route_ways
""").fetchone()[0]

municipality_count = non_bicycle_route_ways_per_municipality.count('*').fetchone()[0]
province_count     = non_bicycle_route_ways_per_province.count('*').fetchone()[0]
h3_count           = non_bicycle_route_ways_per_h3_cell.count('*').fetchone()[0]
 
municipality_length_km = duckdb.sql("""
    SELECT ROUND(SUM(clipped_length_meters) / 1000, 3)
    FROM non_bicycle_route_ways_per_municipality
""").fetchone()[0]
 
province_length_km = duckdb.sql("""
    SELECT ROUND(SUM(clipped_length_meters) / 1000, 3)
    FROM non_bicycle_route_ways_per_province
""").fetchone()[0]
 
h3_length_km = duckdb.sql("""
    SELECT ROUND(SUM(clipped_length_meters) / 1000, 3)
    FROM non_bicycle_route_ways_per_h3_cell
""").fetchone()[0]
 
print(f"{'':40s} {'rows':>10s}  {'multiplier':>10s}  {'length (km)':>12s}")
print(f"{'─'*76}")
print(f"{'Original (bicycle_route_m_ways_distinct)':40s} {original_count:>10,}  {'':>10s}  {original_length_km:>12,.3f}")
print(f"{'After municipality join':40s} {municipality_count:>10,}  {municipality_count/original_count:>10.3f}x  {municipality_length_km:>12,.3f}")
print(f"{'After province join':40s} {province_count:>10,}  {province_count/original_count:>10.3f}x  {province_length_km:>12,.3f}")
print(f"{'After H3 join':40s} {h3_count:>10,}  {h3_count/original_count:>10.3f}x  {h3_length_km:>12,.3f}")

                                               rows  multiplier   length (km)
────────────────────────────────────────────────────────────────────────────
Original (bicycle_route_m_ways_distinct)    249,920                33,159.307
After municipality join                     252,296       1.010x    33,159.307
After province join                         250,093       1.001x    33,159.307
After H3 join                               297,730       1.191x    33,130.201


These differences are expected. The intersection process transforms each original way into one or more records depending on how it overlaps with spatial unit boundaries. Coarser administrative units typically preserve the original structure with limited splitting, while the finer H3 grid induces substantially more segment fragmentation due to its higher spatial resolution.
 
Note that the multiplier is expected to be higher here than in `05_bicycle_route_m_ways_distinct_per_spatial_unit`. Non-route ways include a broader range of geometry types and lengths, including many short local streets, which interact differently with spatial unit boundaries than the longer, corridor-like geometries typical of designated bicycle route ways.

### Export

In [10]:
from pathlib import Path

Path("/local/data/vbo226/cache").mkdir(parents=True, exist_ok=True)

non_bicycle_route_ways_per_municipality.write_parquet(
    "/local/data/vbo226/cache/non_bicycle_route_ways_per_municipality.parquet"
)

non_bicycle_route_ways_per_h3_cell.write_parquet(
    "/local/data/vbo226/cache/non_bicycle_route_ways_per_h3_cell.parquet"
)

non_bicycle_route_ways_per_province.write_parquet(
    "/local/data/vbo226/cache/non_bicycle_route_ways_per_province.parquet"
)